# Config

In [1]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [8]:
import pandas as pd
import os
from preprocess.preprocess import clean_text, check_deleted_expressions
from preprocess.translate import translator, gen_text_for_embedding, final_clean
import time
import json
from sklearn.preprocessing import LabelEncoder

# 1) Preprocesamiento de los datos


In [19]:
# 1) Cargar datos
path = "/tmp/data"
path_analytics = "/tmp/analytics"
filePATH = os.path.join(path, "data_concatenada.xlsx")
df = pd.read_excel(filePATH,
                   usecols=["Código VRID", "Título", "Resumen", "Keywords", "Interdisciplinario", "Transdisciplinario", "Facultad del Proyecto",
                            "Depto Persona"]) \
       .fillna("")

# 2) Guardar qué secuencias de palabras del resumen serán eliminadas al aplicar get_expressions_to_delete()
list_texts = df["Resumen"].to_list()
df_deleted = check_deleted_expressions(list_texts)
savepath=os.path.join(path_analytics, "deleted_re.xlsx")
df_deleted.to_excel(savepath, index=False)

# 3) Preprocesar los datos
#Columnas que se van a preprocesar
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Título": "Titulo_trad",
    "Resumen": "Resumen_trad",
    "Keywords": "keywords_trad",
    "Facultad del Proyecto": "Facultad_del_Proyecto_trad",
    "Depto Persona": "Depto_Persona_trad",
}
#Preprocesamiento de datos
df[list(cols.values())] = df[list(cols.keys())].applymap(clean_text)
savepath=os.path.join(path, "data_clean.xlsx")
df.to_excel(savepath, index=False)

# 2) Traducción del texto

In [21]:
from transformers import MarianMTModel, MarianTokenizer

#1. Cargar modelo de traducción
model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
trans = translator(model, tokenizer)

#Columnas que se van a traducir
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Titulo_trad": "Titulo_trad",
    "Resumen_trad": "Resumen_trad",
    "keywords_trad": "keywords_trad",
    "Facultad_del_Proyecto_trad": "Facultad_del_Proyecto_trad",
    "Depto_Persona_trad": "Depto_Persona_trad",
}

#2. Traducción de columnas
#####Estoy trabajando en mejorar esta parte para que sea más rápida con paralelización por batches
start = time.time()
for src, dst in cols.items():
    df[dst] = trans.translate_parallel(df[src].to_list(), batch_size=8)
end = time.time()


#3.Guardado de resultados
savepath=os.path.join(path, "data_translated.xlsx")
df.to_excel(savepath, index=False)

print(f"Tiempo total de traducción: {end - start:.2f} segundos")

/usr/local/lib/python3.10/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Usando dispositivo: cuda


Traduciendo: 100%|██████████| 119/119 [00:27<00:00,  4.39batch/s]


Tiempo total de traducción: 754.63 segundos


In [28]:
#3. Selección de columnas que se utilizarán en clasificador y concatenación
# Última limpieza antes de generar concatenación
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad", "Facultad_del_Proyecto_trad", "Depto_Persona_trad"]
for col in cols:
    df[col] = df[col].apply(final_clean)

#  Selección de columnas para embedding.
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]
element_names = ["title", "keywords", "abstract"]
df = gen_text_for_embedding(df, cols, element_names)

# Guardado de resultados
savepath=os.path.join(path, "data_translated_concat.xlsx")
df.to_excel(savepath, index=False)
savepath=os.path.join(path, "data_translated_concat.csv")
df.to_csv(savepath, index=False, encoding="utf-8-sig")

df.head()

,Código VRID,Interdisciplinario,Transdisciplinario,Título,Keywords,Resumen,Facultad del Proyecto,Depto Persona,Titulo_trad,Resumen_trad,keywords_trad,Facultad_del_Proyecto_trad,Depto_Persona_trad,text_for_embedding_translated
0,217.173.049-1.0,SI,NaN,PATRONES DE CRIANZA Y SOCIALIZACIÓN DE GÉNERO ...,NaN,OBJETIVOS GENERALES: _x000D_\nDESCRIBIR LOS PR...,FACULTAD DE CIENCIAS SOCIALES,"DEPARTAMENTO DE ECONOMÍA, SIN INFORMACIÓN, ESC...",patterns of gender upbringing and socializatio...,general objectives: to describe the processes ...,,faculty of social sciences,"department of economics, without information, ...",title: patterns of gender upbringing and socia...
1,218.201.002-1.0,SI,NaN,ADAPTACIÓN CULTURAL Y VALIDACIÓN DE LA ESCALA ...,"ESTILO DE VIDA, ADOLESCENTES _x000D_\n",PARA EVALUAR LOS COMPORTAMIENTOS RELACIONADOS ...,FACULTAD DE ENFERMERÍA,"DEPARTAMENTO DE CIENCIA ANIMAL, DEPARTAMENTO D...",cultural adaptation and validation of the life...,in order to evaluate the behaviors related to ...,"lifestyle, teens",faculty of nursing,"department of animal science, department of pl...",title: cultural adaptation and validation of t...
2,218.102.031-1.0IN,NO,NaN,PROMOVIENDO LA REFLEXIÓN EN ESTUDIANTES DE PRE...,NaN,EL PRESENTE PROYECTO INVOLUCRA LA REALIZACIÓN ...,FACULTAD DE ODONTOLOGÍA,DEPARTAMENTO DE ASTRONOMÍA,promoting reflection in preclinical dental stu...,the present project involves the realization o...,,faculty of dentistry,department of astronomy,title: promoting reflection in preclinical den...
3,218.163.016-INI,INDEFINIDO,NaN,MOTIVACIÓN Y HABILIDADES SOCIALES EN ADOLESCENTES,NaN,EL ESTUDIO DE LA MOTIVACIÓN TIENE DIFERENTES A...,FACULTAD DE EDUCACIÓN,"DEPARTAMENTO DE CIENCIAS DE LA EDUCACIÓN, DEPT...",motivation and social skills in adolescents,the study of the motivation has different side...,,faculty of education,"department of education sciences, department o...",title: motivation and social skills in adolesc...
4,219.091.052-INI,NO,NaN,TIME EFFECTS ON THE LIQUEFACTION RESPONSE OF G...,NaN,SECONDARY CONSOLIDATION AND AGEING ARE TWO OFT...,FACULTAD DE INGENIERÍA,DEPTO. TEORÍA POLITICA Y FUND.DE LA EDUC.,time effects on the liquefaction response of g...,secondary consolidation and ageing are two oft...,,faculty of engineering,department of political and fund theory of edu...,title: time effects on the liquefaction respon...


# 3) Split dataset

In [105]:
def to_serializable(obj):
    if hasattr(obj, "tolist"):
        return obj.tolist()
    return obj

In [109]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
import json

path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

df = df[df["Interdisciplinario"] != "INDEFINIDO"]
le = LabelEncoder()
df["labels"] = le.fit_transform(df["Interdisciplinario"])

# Convertir a arrays
ids = df["Código VRID"].to_numpy()
labels = df["labels"].to_numpy()

# Train/Test split (ids y labels en paralelo)
idx_train, idx_test, y_train, y_test = train_test_split(
    ids,
    labels,
    test_size=0.2,
    random_state=7,
    stratify=labels
)

# Crear folds sobre train
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)

folds = []
for fold, (train_pos, val_pos) in enumerate(skf.split(idx_train, y_train)):
    train_ids = idx_train[train_pos]   # array de IDs
    val_ids = idx_train[val_pos]       # array de IDs
    folds.append(val_ids)

print("Test size:", len(idx_test))
print("Fold 0 - Val size:", len(folds[0]))

#Guardar index en diccionario
dataset_index = {
    "Train": idx_train,
    "Test": idx_test,
    "kfolds": folds 
}
filepath=os.path.join(path, "train_test_ids_3folds.json")

# Guardar
with open(filepath, "w", encoding="utf-8") as f:
    json.dump(dataset_index, f, default=to_serializable, indent=2, ensure_ascii=False)


Test size: 193
Fold 0 - Val size: 257


# 4) TF-ID feature extractor 

## Functions

In [38]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

#Creacion de vectores TFID
from sklearn.feature_extraction.text import TfidfVectorizer

# Descargar recursos necesarios (solo la primera vez)
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

def preprocess_text_for_TFID(texts):
    """
    Preprocesa textos en inglés para TF-IDF:
    - Minúsculas
    - Tokenización con regex
    - Eliminación de stopwords
    - Lematización
    """
    stop_words = set(stopwords.words("english"))
    lemmatizer = WordNetLemmatizer()
    processed_texts = []

    for text in texts:
        # Minúsculas
        text = text.lower()

        # Tokenización: mantener solo palabras (a-z)
        tokens = re.findall(r"\b[a-z]+\b", text)

        # Eliminar stopwords y lematizar
        tokens = [
            lemmatizer.lemmatize(token) 
            for token in tokens if token not in stop_words
        ]

        processed_texts.append(" ".join(tokens))
    
    return processed_texts

def code_to_idx(list_codes, df):
    """
    Retorna los índices de las filas del DataFrame df 
    en las que la columna 'Código VRID' coincide con 
    alguno de los códigos en list_codes.

    Parámetros
    ----------
    list_codes : list
        Lista de códigos a buscar.
    df : pandas.DataFrame
        DataFrame que contiene la columna 'Código VRID'.

    Retorna
    -------
    list
        Lista con los índices del DataFrame correspondientes a los códigos encontrados.
    """
    mask = df["Código VRID"].isin(list_codes)
    return df.index[mask].tolist()

def gen_TFID_vectors(texts):
    # Crear el vectorizador
    vectorizer = TfidfVectorizer()

    # Ajustar y transformar los documentos
    X = vectorizer.fit_transform(texts)

    return X


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, make_scorer

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

import copy
from sklearn.model_selection import train_test_split

from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

from sklearn.decomposition import PCA

from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
import numpy as np

#import mlflow

def eval_model(best_model, X_test, y_test):
  results = {}
  preds = best_model.predict(X_test)
  cm = confusion_matrix(y_test, preds)
  t_n, f_p, f_n, t_p = cm.ravel()
  results = {
      'accuracy': accuracy_score(y_test, preds),
      'precision': precision_score(y_test, preds, zero_division=0),
      'recall': recall_score(y_test, preds, zero_division=0),
      'f1_score': f1_score(y_test, preds, zero_division=0),
      't_n': t_n,
      'f_p': f_p,
      'f_n': f_n,
      't_p': t_p
  }
  return results

#MLflow logging helper function
def safe_log_metric(name, value):
    try:
        if isinstance(value, (list, tuple, np.ndarray)):
            if np.size(value) == 1:
                value = float(np.array(value).item())
            else:
                raise ValueError("Métrica con más de un valor.")
        else:
            value = float(value)
        mlflow.log_metric(name, value)
    except Exception as e:
        print(f"⚠️ No se pudo loggear {name}: {e}")

def mlflow_ckeckpoint(results_val, models_dicc, X_test, y_test, experiment_name):
    # Define el experimento (lo crea si no existe) 
    mlflow.set_tracking_uri("sqlite:///mlflow.db")
    mlflow.set_experiment(experiment_name)

    for model_name, metrics in results_val.items():
        model = models_dicc[model_name]

        with mlflow.start_run(run_name=model_name):
            print(f"Registrando modelo en MLflow: {model_name}")

            # Hiperparámetros
            try:
                mlflow.log_params(model.get_params())
            except:
                print(f"No se pudieron loggear los hiperparámetros para {model_name}")

            # Métricas de validación
            for k, v in metrics.items():
                safe_log_metric(f"val_{k}", v)

            # Métricas de test
            results_test = eval_model(model, X_test, y_test)
            for k, v in results_test.items():
                safe_log_metric(f"test_{k}", v)

            # Guardar modelo
            mlflow.sklearn.log_model(model, name = "model", input_example=X_test[:5])  

#Pipeline helper functions
def get_est_params_dict(keys):
    clf_params_dict = {
        'LogisticRegression': {
            'class': LogisticRegression,
            'params': {
                'solver': Categorical(['lbfgs']),
                'penalty': Categorical(['l2']),
                'C': [0.1, 1.0, 10]
            }
        },
        'RandomForestClassifier': {
            'class': RandomForestClassifier,
            'params': {
                'bootstrap': Categorical([True]),
                'n_estimators': Integer(40, 300),
                'max_depth': Integer(3, 14),
                'min_samples_split': Integer(2, 5),
                'min_samples_leaf': Integer(1, 2),
            }
        },
        'GradientBoostingClassifier': {
            'class': GradientBoostingClassifier,
            'params': {
                'n_estimators': Integer(50, 200),
                'learning_rate': Real(0.01, 0.2),
                'max_depth': Integer(3, 7)
            }
        },
        'XGBClassifier': {
            'class': XGBClassifier,
            'params': {
                'n_estimators': Integer(40, 300),
                'learning_rate': Real(0.01, 0.2),
                'max_depth': Integer(3, 14),
                'eval_metric': Categorical(['logloss'])
            }
        },

    }

    # Filtrar y retornar solo los modelos solicitados
    return {key: clf_params_dict[key] for key in keys if key in clf_params_dict}

def setup_model(dicc):
    model_ = dicc['class']()
    #scaler = StandardScaler()
    model = Pipeline([('model', model_)])
    param_grid = {'model__' + param_name: param_value for param_name, param_value in dicc['params'].items()}
    return model, param_grid

def run_BayesSearchCV(model, param_grid, X_train, y_train, n_iter=10, scoring='recall', sample_weight=None):
    from sklearn.exceptions import FitFailedWarning
    import warnings

    bayes_searchCV = BayesSearchCV(
        estimator=model,
        search_spaces=param_grid,
        cv=5,
        scoring=scoring,
        n_iter = n_iter,
        n_jobs=4,
        n_points = 2
    )

    try:
        fit_params = {'model__sample_weight': sample_weight} if sample_weight is not None else {}
        bayes_searchCV.fit(X_train, y_train, **fit_params)
    except TypeError as e:
        print(f"⚠️ Modelo {model.named_steps['model'].__class__.__name__} no acepta sample_weight. Reintentando sin él.")
        bayes_searchCV.fit(X_train, y_train)

    return bayes_searchCV

def calculate_metrics(y_true, y_pred, y_proba=None):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
    }

    # AUROC y AUPRC requieren probabilidades (y_proba)
    if y_proba is not None:
        try:
            metrics["auroc"] = roc_auc_score(y_true, y_proba)
            metrics["auprc"] = average_precision_score(y_true, y_proba)
        except:
            metrics["auroc"] = np.nan
            metrics["auprc"] = np.nan

    return metrics

def get_sample_weight(y_train):
    """
    Devuelve un array de sample weights donde las muestras con y=1
    tienen 10 veces más peso que las muestras con y=0.
    """
    y_train = np.asarray(y_train)
    weights = np.ones_like(y_train, dtype=np.float64)
    weights[y_train == 1] = 10
    return weights

def select_best_model(results_val, models_dicc):
    best_result = 0
    best_model = None
    for model, metrics in results_val.items():
        if metrics['mean_test_score'] > best_result:
            best_result = metrics['mean_test_score']
            best_model = models_dicc[model]
    return best_model

#Pipeline function to run the entire ML pipeline
def run_bayesian_pipeline(est_params_dict, data, labels, n_iter, sample_weight_On = None):

    # creacion de diccionarios para almacenamiento
    results_test = {}
    cm_test = {}
    results_val = {}
    models_dicc = {}

    for model_name, dicc in est_params_dict.items():

        print(model_name)
        model, param_grid = setup_model(dicc)

        #Sample weights strategy
        if sample_weight_On is not None:
          print('Compute sw')
          sample_weight = compute_sample_weight(class_weight='balanced', y=labels)
        else:
          sample_weight = None
        grid_search = run_BayesSearchCV(model, param_grid, data, labels, n_iter = n_iter, scoring = 'recall', sample_weight = sample_weight)


        # Guardar best model
        best_model = grid_search.best_estimator_
        models_dicc[model_name] = copy.deepcopy(best_model)

        mean_val_score = grid_search.cv_results_['mean_test_score'][grid_search.best_index_]
        std_val_score = grid_search.cv_results_['std_test_score'][grid_search.best_index_]
        mean_val_score = np.round(mean_val_score,2)
        std_val_score = np.round(std_val_score,2)

        metrics_val = {
          'mean_test_score': float(np.round(mean_val_score, 2)),
          'std_test_score': float(np.round(std_val_score, 2)
        )}

        results_val[model_name]  = metrics_val


    return results_val, models_dicc


# Ejemplo de uso

# 1. Generar dataset sintético
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_repeated=0,
    n_classes=2,
    #weights=[0.95],  # 90% clase 0, 10% clase 1
    random_state=42,
    shuffle=True
)

# 2. Elegir modelos a probar
model_keys = [
    #'LogisticRegression',
    #'DecisionTreeClassifier',
    #'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    #'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 Comienzo:", Counter(y))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X, y, n_iter=20, sample_weight_On = True)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 Comienzo: Counter({1: 503, 0: 497})
XGBClassifier
Compute sw

🔍 Validación:
XGBClassifier: {'mean_test_score': 0.93, 'std_test_score': 0.03}


## Code

Cargar data

In [39]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

Generar dataset para clasificación

In [94]:
# Read data
df = df[df["Interdisciplinario"]!="INDEFINIDO"]
X = df["text_for_embedding_translated"]
y = df["Interdisciplinario"]
#Lemantización y eliminación de stopwords
X = preprocess_text_for_TFID(X)
# Codificación de labels
le = LabelEncoder()
y = le.fit_transform(y)
#Creacion de vectores TFID
X = gen_TFID_vectors(X)

Obtener index de datos para clasificación

In [103]:
def code_to_idx(list_codes, df):
    """
    Retorna los índices de las filas del DataFrame df 
    en las que la columna 'Código VRID' coincide con 
    alguno de los códigos en list_codes.

    Parámetros
    ----------
    list_codes : list
        Lista de códigos a buscar.
    df : pandas.DataFrame
        DataFrame que contiene la columna 'Código VRID'.

    Retorna
    -------
    list
        Lista con los índices del DataFrame correspondientes a los códigos encontrados.
    """
    mask = df["Código VRID"].isin(list_codes)
    return df.index[mask].tolist()

class CvCustom():
    def __init__(self, n_splits = None):
        #Cargar lista de códigos VRID
        path = "/tmp/data"
        filepath=os.path.join(path, "train_test_ids_3folds.json")
        with open(filepath, "r", encoding="utf-8") as f:
            dataset_index = json.load(f)
        #Cagar base de datos
        path = "/tmp/data"
        filepath=os.path.join(path, "data_translated_concat.csv")
        self.df = pd.read_csv(filepath)
        self.df = df[df["Interdisciplinario"]!="INDEFINIDO"]
        #Obtener cantidad de folds
        self.n_splits = len(dataset_index["kfolds"])
        #Transformar a idx y almacenarlos
        self.kfolds = []
        for i in range(self.n_splits):
            kfold = dataset_index["kfolds"][i]
            self.kfolds.append(kfold)
        
    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    def split(self, X, y=None, groups=None):
        for i in range(self.n_splits):
            test_idx = self.kfolds[i]["val_idx"]
            test_idx = np.array(code_to_idx(test_idx, self.df))
            train_idx = self.kfolds[i]["train_idx"]
            train_idx = np.array(code_to_idx(train_idx, self.df))
            print(train_idx.max(), test_idx.max())
            yield train_idx, test_idx

skf = CvCustom()
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    print(fold, train_idx.shape, test_idx.shape)


1081 1082
1 (514,) (257,)
1082 1081
2 (514,) (257,)
1082 1080
3 (514,) (257,)


In [102]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, make_scorer

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

import copy
from sklearn.model_selection import train_test_split

from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

from sklearn.decomposition import PCA

from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
import numpy as np

#import mlflow

def eval_model(best_model, X_test, y_test):
  results = {}
  preds = best_model.predict(X_test)
  cm = confusion_matrix(y_test, preds)
  t_n, f_p, f_n, t_p = cm.ravel()
  results = {
      'accuracy': accuracy_score(y_test, preds),
      'precision': precision_score(y_test, preds, zero_division=0),
      'recall': recall_score(y_test, preds, zero_division=0),
      'f1_score': f1_score(y_test, preds, zero_division=0),
      't_n': t_n,
      'f_p': f_p,
      'f_n': f_n,
      't_p': t_p
  }
  return results

#MLflow logging helper function
def safe_log_metric(name, value):
    try:
        if isinstance(value, (list, tuple, np.ndarray)):
            if np.size(value) == 1:
                value = float(np.array(value).item())
            else:
                raise ValueError("Métrica con más de un valor.")
        else:
            value = float(value)
        mlflow.log_metric(name, value)
    except Exception as e:
        print(f"⚠️ No se pudo loggear {name}: {e}")

def mlflow_ckeckpoint(results_val, models_dicc, X_test, y_test, experiment_name):
    # Define el experimento (lo crea si no existe) 
    mlflow.set_tracking_uri("sqlite:///mlflow.db")
    mlflow.set_experiment(experiment_name)

    for model_name, metrics in results_val.items():
        model = models_dicc[model_name]

        with mlflow.start_run(run_name=model_name):
            print(f"Registrando modelo en MLflow: {model_name}")

            # Hiperparámetros
            try:
                mlflow.log_params(model.get_params())
            except:
                print(f"No se pudieron loggear los hiperparámetros para {model_name}")

            # Métricas de validación
            for k, v in metrics.items():
                safe_log_metric(f"val_{k}", v)

            # Métricas de test
            results_test = eval_model(model, X_test, y_test)
            for k, v in results_test.items():
                safe_log_metric(f"test_{k}", v)

            # Guardar modelo
            mlflow.sklearn.log_model(model, name = "model", input_example=X_test[:5])  

#Pipeline helper functions
def get_est_params_dict(keys):
    clf_params_dict = {
        'LogisticRegression': {
            'class': LogisticRegression,
            'params': {
                'solver': Categorical(['lbfgs']),
                'penalty': Categorical(['l2']),
                'C': [0.1, 1.0, 10]
            }
        },
        'RandomForestClassifier': {
            'class': RandomForestClassifier,
            'params': {
                'bootstrap': Categorical([True]),
                'n_estimators': Integer(40, 300),
                'max_depth': Integer(3, 14),
                'min_samples_split': Integer(2, 5),
                'min_samples_leaf': Integer(1, 2),
            }
        },
        'GradientBoostingClassifier': {
            'class': GradientBoostingClassifier,
            'params': {
                'n_estimators': Integer(50, 200),
                'learning_rate': Real(0.01, 0.2),
                'max_depth': Integer(3, 7)
            }
        },
        'XGBClassifier': {
            'class': XGBClassifier,
            'params': {
                'n_estimators': Integer(40, 300),
                'learning_rate': Real(0.01, 0.2),
                'max_depth': Integer(3, 14),
                'eval_metric': Categorical(['logloss'])
            }
        },

    }

    # Filtrar y retornar solo los modelos solicitados
    return {key: clf_params_dict[key] for key in keys if key in clf_params_dict}

def setup_model(dicc):
    model_ = dicc['class']()
    #scaler = StandardScaler()
    model = Pipeline([('model', model_)])
    param_grid = {'model__' + param_name: param_value for param_name, param_value in dicc['params'].items()}
    return model, param_grid

def run_BayesSearchCV(model, param_grid, X_train, y_train, n_iter=10, scoring='recall', sample_weight=None):
    from sklearn.exceptions import FitFailedWarning
    import warnings

    bayes_searchCV = BayesSearchCV(
        estimator=model,
        search_spaces=param_grid,
        cv=CvCustom(),
        scoring=scoring,
        n_iter = n_iter,
        n_jobs=4,
        n_points = 2
    )

    try:
        fit_params = {'model__sample_weight': sample_weight} if sample_weight is not None else {}
        bayes_searchCV.fit(X_train, y_train, **fit_params)
    except TypeError as e:
        print(f"⚠️ Modelo {model.named_steps['model'].__class__.__name__} no acepta sample_weight. Reintentando sin él.")
        bayes_searchCV.fit(X_train, y_train)

    return bayes_searchCV

def calculate_metrics(y_true, y_pred, y_proba=None):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
    }

    # AUROC y AUPRC requieren probabilidades (y_proba)
    if y_proba is not None:
        try:
            metrics["auroc"] = roc_auc_score(y_true, y_proba)
            metrics["auprc"] = average_precision_score(y_true, y_proba)
        except:
            metrics["auroc"] = np.nan
            metrics["auprc"] = np.nan

    return metrics

def get_sample_weight(y_train):
    """
    Devuelve un array de sample weights donde las muestras con y=1
    tienen 10 veces más peso que las muestras con y=0.
    """
    y_train = np.asarray(y_train)
    weights = np.ones_like(y_train, dtype=np.float64)
    weights[y_train == 1] = 10
    return weights

def select_best_model(results_val, models_dicc):
    best_result = 0
    best_model = None
    for model, metrics in results_val.items():
        if metrics['mean_test_score'] > best_result:
            best_result = metrics['mean_test_score']
            best_model = models_dicc[model]
    return best_model

#Pipeline function to run the entire ML pipeline
def run_bayesian_pipeline(est_params_dict, data, labels, n_iter, sample_weight_On = None):

    # creacion de diccionarios para almacenamiento
    results_test = {}
    cm_test = {}
    results_val = {}
    models_dicc = {}

    for model_name, dicc in est_params_dict.items():

        print(model_name)
        model, param_grid = setup_model(dicc)

        #Sample weights strategy
        if sample_weight_On is not None:
          print('Compute sw')
          sample_weight = compute_sample_weight(class_weight='balanced', y=labels)
        else:
          sample_weight = None
        grid_search = run_BayesSearchCV(model, param_grid, data, labels, n_iter = n_iter, scoring = 'recall', sample_weight = sample_weight)


        # Guardar best model
        best_model = grid_search.best_estimator_
        models_dicc[model_name] = copy.deepcopy(best_model)

        mean_val_score = grid_search.cv_results_['mean_test_score'][grid_search.best_index_]
        std_val_score = grid_search.cv_results_['std_test_score'][grid_search.best_index_]
        mean_val_score = np.round(mean_val_score,2)
        std_val_score = np.round(std_val_score,2)

        metrics_val = {
          'mean_test_score': float(np.round(mean_val_score, 2)),
          'std_test_score': float(np.round(std_val_score, 2)
        )}

        results_val[model_name]  = metrics_val


    return results_val, models_dicc


# 2. Elegir modelos a probar
model_keys = [
    #'LogisticRegression',
    #'DecisionTreeClassifier',
    #'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    #'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 Comienzo:", Counter(y))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X, y, n_iter=2, sample_weight_On = True)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 Comienzo: Counter({1: 555, 0: 409})
XGBClassifier
Compute sw


IndexError: index 966 is out of bounds for axis 0 with size 964

In [100]:
X.shape, y.shape

((964, 18390), (964,))

# Code 2

In [140]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [ ]:
# Read data
df = df[df["Interdisciplinario"]!="INDEFINIDO"]
X = df["text_for_embedding_translated"]
y = df["Interdisciplinario"]
#Lemantización y eliminación de stopwords
X = preprocess_text_for_TFID(X)
# Codificación de labels
le = LabelEncoder()
y = le.fit_transform(y)
#Creacion de vectores TFID
X = gen_TFID_vectors(X)

In [ ]:
def code_to_idx(list_codes, df):
    """
    Retorna los índices de las filas del DataFrame df 
    en las que la columna 'Código VRID' coincide con 
    alguno de los códigos en list_codes.

    Parámetros
    ----------
    list_codes : list
        Lista de códigos a buscar.
    df : pandas.DataFrame
        DataFrame que contiene la columna 'Código VRID'.

    Retorna
    -------
    list
        Lista con los índices del DataFrame correspondientes a los códigos encontrados.
    """
    mask = df["Código VRID"].isin(list_codes)
    idx = df.index[mask].tolist()
    df_deco = pd.DataFrame({
    "idx": idx,
    "code": list_codes
    })
    return df_deco

def select_data(list_codes, df):
    mask = df["Código VRID"].isin(list_codes)
    new_df = df[mask]
    #Gen idx
    idx = df.index[df["Código VRID"].isin(list_codes)]
    df_deco = pd.DataFrame({
    "idx": idx,
    "code": list_codes,
    })
    #Ordenar los datos por index
    df_deco = df_deco.sort_values(by="idx").reset_index(drop=True)
    return new_df, df_deco

#Codificación de IDs: Se considera orden en el que IDs se presentan en el dataframe.
codes_vrid = dataset_index["kfolds"]
codes_vrid = np.array([i for fold in codes_vrid for i in fold])
new_df, df_deco = select_data(codes_vrid, df)


,idx,code
0,1,ID24I10166
1,4,3240234
2,5,FOVI220125
3,6,2022000497INV
4,7,2021000207INI
...,...,...
766,1075,BIP40040959-0
767,1078,ID21I10075
768,1080,ID23I10099
769,1081,1221600


In [ ]:
class CvCustom():
    def __init__(self, n_splits = None):
        #Codificación de IDs: Se considera orden en el que IDs se presentan en el dataframe.
        codes_vrid = dataset_index["kfolds"]
        codes_vrid = np.array([i for fold in codes_vrid for i in fold])
        new_df, df_deco = select_data(codes_vrid, df)
            
    
    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    def split(self, X, y=None, groups=None):
        for i in range(self.n_splits):
            test_idx = self.kfolds[i]["val_idx"]
            test_idx = np.array(code_to_idx(test_idx, self.df))
            train_idx = self.kfolds[i]["train_idx"]
            train_idx = np.array(code_to_idx(train_idx, self.df))
            print(train_idx.max(), test_idx.max())
            yield train_idx, test_idx

,idx,code
0,0,ID24I10166
1,1,3240234
2,2,FOVI220125
3,3,2022000497INV
4,4,2021000207INI
...,...,...
766,766,BIP40040959-0
767,767,ID21I10075
768,768,ID23I10099
769,769,1221600


# Code 3

In [213]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [237]:
def gen_TFID_dataset(codes_vrid, df):
    #Selección unicamente de elementos de df que se encuentren en codes_vrid
    df = df[df["Código VRID"].isin(codes_vrid)].copy()

    #Creación de index en función de orden de los datos
    df['idx'] = np.arange(0, df.shape[0])

    #Generación de datasets
    X = df["text_for_embedding_translated"]
    y = df["Interdisciplinario"]

    #Lemantización y eliminación de stopwords
    X = preprocess_text_for_TFID(X)
    # Codificación de labels
    le = LabelEncoder()
    y = le.fit_transform(y)
    #Creacion de vectores TFID
    X = gen_TFID_vectors(X)

    return X, y, df
    
#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, _ = gen_TFID_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, new_df = gen_TFID_dataset(codes_train, df)

#Diccionario de decodificación
df_decode = new_df[["idx", "Código VRID"]]

In [249]:
def gen_TFID_dataset(codes_vrid, df):
    #Selección unicamente de elementos de df que se encuentren en codes_vrid
    df = df[df["Código VRID"].isin(codes_vrid)].copy()

    #Creación de index en función de orden de los datos
    df['idx'] = np.arange(0, df.shape[0])

    #Generación de datasets
    X = df["text_for_embedding_translated"]
    y = df["Interdisciplinario"]
    return X, y

def gen_TFID_vectors(train, test):
    # Crear el vectorizador
    vectorizer = TfidfVectorizer()

    # Ajustar y transformar los documentos
    X_train = vectorizer.fit_transform(train)
    X_test = vectorizer.transform(test)

    return X_train, X_test

    
#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test= gen_TFID_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train= gen_TFID_dataset(codes_train, df)

#Lemantización y eliminación de stopwords
X_train = preprocess_text_for_TFID(X_train)
# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)
#Creacion de vectores TFID
X_train, X_test = gen_TFID_vectors(X_train, X_test)


#Diccionario de decodificación
#df_decode = new_df[["idx", "Código VRID"]]

In [ ]:
def decode_codes(fold_codes, df_decode):
    """
    Decodifica fold_codes usando df_decode.
    fold_codes : lista o array con índices (ej. [0, 2, 5])
    df_decode  : DataFrame con columnas ['idx', 'Código VRID']

    Devuelve un numpy.array con los códigos VRID correspondientes.
    """
    # Crear un diccionario {Código VRID: código}
    mapping = df_decode.set_index("Código VRID")["idx"].to_dict()

    # Mapear los fold_codes a códigos (ignora los que no existan en mapping)
    decoded = [mapping[c] for c in fold_codes if c in mapping]

    return np.array(decoded)

class CvCustom():
    def __init__(self, df_decode, n_splits = None):
        #Dict codes
        self.df_decode=df_decode
        #Lectura de index de separacion de conjuntos train/test
        path = "/tmp/data"
        filepath=os.path.join(path, "train_test_ids_3folds.json")
        with open(filepath, "r", encoding="utf-8") as f:
            dataset_index = json.load(f)
        folds_codes = dataset_index["kfolds"]
        self.n_splits=len(folds_codes)
        #Define index for kfolds
        self.kfolds = []
        for i in range(self.n_splits):
            self.kfolds.append(decode_codes(folds_codes[i], self.df_decode))
            
        #Save al idx
        self.all_idx = np.array([i for fold in self.kfolds for i in fold])
        print(self.all_idx.shape)
    
    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    def split(self, X, y=None, groups=None):
        for i in range(self.n_splits):
            test_idx = self.kfolds[i]
            train_idx = np.setdiff1d(self.all_idx, test_idx) 
            yield train_idx, test_idx


(771,)
1 (514,) (257,)
2 (514,) (257,)
3 (514,) (257,)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, make_scorer

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

import copy
from sklearn.model_selection import train_test_split

from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

from sklearn.decomposition import PCA

from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
import numpy as np

#import mlflow

def eval_model(best_model, X_test, y_test):
  results = {}
  preds = best_model.predict(X_test)
  cm = confusion_matrix(y_test, preds)
  t_n, f_p, f_n, t_p = cm.ravel()
  results = {
      'accuracy': accuracy_score(y_test, preds),
      'precision': precision_score(y_test, preds, zero_division=0),
      'recall': recall_score(y_test, preds, zero_division=0),
      'f1_score': f1_score(y_test, preds, zero_division=0),
      't_n': t_n,
      'f_p': f_p,
      'f_n': f_n,
      't_p': t_p
  }
  return results

#MLflow logging helper function
def safe_log_metric(name, value):
    try:
        if isinstance(value, (list, tuple, np.ndarray)):
            if np.size(value) == 1:
                value = float(np.array(value).item())
            else:
                raise ValueError("Métrica con más de un valor.")
        else:
            value = float(value)
        mlflow.log_metric(name, value)
    except Exception as e:
        print(f"⚠️ No se pudo loggear {name}: {e}")

def mlflow_ckeckpoint(results_val, models_dicc, X_test, y_test, experiment_name):
    # Define el experimento (lo crea si no existe) 
    mlflow.set_tracking_uri("sqlite:///mlflow.db")
    mlflow.set_experiment(experiment_name)

    for model_name, metrics in results_val.items():
        model = models_dicc[model_name]

        with mlflow.start_run(run_name=model_name):
            print(f"Registrando modelo en MLflow: {model_name}")

            # Hiperparámetros
            try:
                mlflow.log_params(model.get_params())
            except:
                print(f"No se pudieron loggear los hiperparámetros para {model_name}")

            # Métricas de validación
            for k, v in metrics.items():
                safe_log_metric(f"val_{k}", v)

            # Métricas de test
            results_test = eval_model(model, X_test, y_test)
            for k, v in results_test.items():
                safe_log_metric(f"test_{k}", v)

            # Guardar modelo
            mlflow.sklearn.log_model(model, name = "model", input_example=X_test[:5])  

#Pipeline helper functions
def get_est_params_dict(keys):
    clf_params_dict = {
        'LogisticRegression': {
            'class': LogisticRegression,
            'params': {
                'solver': Categorical(['lbfgs']),
                'penalty': Categorical(['l2']),
                'C': [0.1, 1.0, 10]
            }
        },
        'RandomForestClassifier': {
            'class': RandomForestClassifier,
            'params': {
                'bootstrap': Categorical([True]),
                'n_estimators': Integer(40, 300),
                'max_depth': Integer(3, 14),
                'min_samples_split': Integer(2, 5),
                'min_samples_leaf': Integer(1, 2),
            }
        },
        'GradientBoostingClassifier': {
            'class': GradientBoostingClassifier,
            'params': {
                'n_estimators': Integer(50, 200),
                'learning_rate': Real(0.01, 0.2),
                'max_depth': Integer(3, 7)
            }
        },
        'XGBClassifier': {
            'class': XGBClassifier,
            'params': {
                'n_estimators': Integer(40, 300),
                'learning_rate': Real(0.01, 0.2),
                'max_depth': Integer(3, 14),
                'eval_metric': Categorical(['logloss'])
            }
        },
        'SVC': {
            'class': SVC,
            'params': {
                'C': Real(1e-3, 1e3, prior='log-uniform'),          # penalización
                'kernel': Categorical(['linear', 'rbf', 'poly']),   # tipos de kernel
                'degree': Integer(2, 5),                            # usado solo si kernel='poly'
                'gamma': Real(1e-4, 1e1, prior='log-uniform'),      # solo para 'rbf','poly','sigmoid'
                'coef0': Real(0.0, 1.0),                            # usado en 'poly' y 'sigmoid'
            }
        }

    }

    # Filtrar y retornar solo los modelos solicitados
    return {key: clf_params_dict[key] for key in keys if key in clf_params_dict}

def setup_model(dicc):
    model_ = dicc['class']()
    #scaler = StandardScaler()
    model = Pipeline([('model', model_)])
    param_grid = {'model__' + param_name: param_value for param_name, param_value in dicc['params'].items()}
    return model, param_grid

def run_BayesSearchCV(model, param_grid, X_train, y_train, n_iter=10, scoring='f1_weighted', sample_weight=None):
    from sklearn.exceptions import FitFailedWarning
    import warnings


    bayes_searchCV = BayesSearchCV(
        estimator=model,
        search_spaces=param_grid,
        cv=CvCustom(df_decode=df_decode),
        scoring=scoring,
        n_iter = n_iter,
        n_jobs=4,
        n_points = 2
    )

    try:
        fit_params = {'model__sample_weight': sample_weight} if sample_weight is not None else {}
        bayes_searchCV.fit(X_train, y_train, **fit_params)
    except TypeError as e:
        print(f"⚠️ Modelo {model.named_steps['model'].__class__.__name__} no acepta sample_weight. Reintentando sin él.")
        bayes_searchCV.fit(X_train, y_train)

    return bayes_searchCV

def calculate_metrics(y_true, y_pred, y_proba=None):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
    }

    # AUROC y AUPRC requieren probabilidades (y_proba)
    if y_proba is not None:
        try:
            metrics["auroc"] = roc_auc_score(y_true, y_proba)
            metrics["auprc"] = average_precision_score(y_true, y_proba)
        except:
            metrics["auroc"] = np.nan
            metrics["auprc"] = np.nan

    return metrics

def get_sample_weight(y_train):
    """
    Devuelve un array de sample weights donde las muestras con y=1
    tienen 10 veces más peso que las muestras con y=0.
    """
    y_train = np.asarray(y_train)
    weights = np.ones_like(y_train, dtype=np.float64)
    weights[y_train == 1] = 10
    return weights

def select_best_model(results_val, models_dicc):
    best_result = 0
    best_model = None
    for model, metrics in results_val.items():
        if metrics['mean_test_score'] > best_result:
            best_result = metrics['mean_test_score']
            best_model = models_dicc[model]
    return best_model

#Pipeline function to run the entire ML pipeline
def run_bayesian_pipeline(est_params_dict, data, labels, n_iter, sample_weight_On = None):

    # creacion de diccionarios para almacenamiento
    results_test = {}
    cm_test = {}
    results_val = {}
    models_dicc = {}

    for model_name, dicc in est_params_dict.items():

        print(model_name)
        model, param_grid = setup_model(dicc)

        #Sample weights strategy
        if sample_weight_On is not None:
          print('Compute sw')
          sample_weight = compute_sample_weight(class_weight='balanced', y=labels)
        else:
          sample_weight = None
        grid_search = run_BayesSearchCV(model, param_grid, data, labels, n_iter = n_iter, scoring = 'f1_weighted', sample_weight = sample_weight)


        # Guardar best model
        best_model = grid_search.best_estimator_
        models_dicc[model_name] = copy.deepcopy(best_model)

        mean_val_score = grid_search.cv_results_['mean_test_score'][grid_search.best_index_]
        std_val_score = grid_search.cv_results_['std_test_score'][grid_search.best_index_]
        mean_val_score = np.round(mean_val_score,2)
        std_val_score = np.round(std_val_score,2)

        metrics_val = {
          'mean_test_score': float(np.round(mean_val_score, 2)),
          'std_test_score': float(np.round(std_val_score, 2)
        )}

        results_val[model_name]  = metrics_val


    return results_val, models_dicc


# 2. Elegir modelos a probar
model_keys = [
    #'LogisticRegression',
    #'DecisionTreeClassifier',
    'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, n_iter=30, sample_weight_On = True)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
RandomForestClassifier
Compute sw
(771,)


XGBClassifier
Compute sw
(771,)
SVC
Compute sw
(771,)

🔍 Validación:
RandomForestClassifier: {'mean_test_score': 0.65, 'std_test_score': 0.03}
XGBClassifier: {'mean_test_score': 0.63, 'std_test_score': 0.02}
SVC: {'mean_test_score': 0.63, 'std_test_score': 0.01}


In [284]:
def eval_model(best_model, X_test, y_test):
  results = {}
  preds = best_model.predict(X_test)
  cm = confusion_matrix(y_test, preds)
  #t_n, f_p, f_n, t_p = cm()
  results = {
      'accuracy': accuracy_score(y_test, preds),
      'precision': precision_score(y_test, preds, zero_division=0),
      'recall': recall_score(y_test, preds, zero_division=0),
      'f1_score': f1_score(y_test, preds, zero_division=0),
       'cm': cm
  }
  return results



for name, model in models_dicc.items():
    results=eval_model(model, X_test, y_test)
    print(results)

{'accuracy': 0.6683937823834197, 'precision': 0.6715328467153284, 'recall': 0.8288288288288288, 'f1_score': 0.7419354838709676, 'cm': array([[37, 45],
       [19, 92]])}
{'accuracy': 0.6217616580310881, 'precision': 0.6461538461538462, 'recall': 0.7567567567567568, 'f1_score': 0.6970954356846473, 'cm': array([[36, 46],
       [27, 84]])}
{'accuracy': 0.6787564766839378, 'precision': 0.7474747474747475, 'recall': 0.6666666666666666, 'f1_score': 0.7047619047619048, 'cm': array([[57, 25],
       [37, 74]])}


In [285]:
import pickle
import os

results_val = {}

for name, model in models_dicc.items():
    results = eval_model(model, X_test, y_test)
    results_val[name] = results
    
    # Guardar cada modelo con pickle
    model_path = os.path.join(path, f"{name}_model.pkl")
    with open(model_path, "wb") as f:
        pickle.dump(model, f)

# Guardar métricas en pickle
results_path = os.path.join(path, "results.pkl")
with open(results_path, "wb") as f:
    pickle.dump(results_val, f)
